# Eggs collected by collector

This notebook totals `Individual Count` for every collector in the egg-slip CSV. It normalizes punctuation and name order, splits named co-collectors, applies a conservative editable alias table, and saves two bar charts plus audit CSVs.

When a clutch has multiple named collectors, the full clutch is credited to each person. Therefore collector-credit totals can exceed the physical number of eggs in the collection.

In [ ]:
from pathlib import Path
from collections import Counter, defaultdict
from difflib import SequenceMatcher
from itertools import combinations
import math
import re
import unicodedata

import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.ticker import LogLocator, FuncFormatter

CSV_PATH = Path(r"G:\My Drive\Egg Slip Scanning\EggSlipReorganizationProject_FULL.xlsx - Full List.csv")
OUTPUT_DIR = Path(r"G:\My Drive\Egg Slip Scanning\scripts\graphs")

# This fallback exists only so the attached copy can be tested in ChatGPT.
ATTACHED_CSV = Path(r"/workspace/scratch/038a5741fbae/upload/01-EggSlipReorganizationProject_FULL.xlsx-Full-List.csv")
if not CSV_PATH.exists() and ATTACHED_CSV.exists():
    CSV_PATH = ATTACHED_CSV
    OUTPUT_DIR = Path.cwd() / "collector_graph_output"

if not CSV_PATH.exists():
    raise FileNotFoundError(f"CSV not found: {CSV_PATH}")

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
print(f"Reading: {CSV_PATH}")
print(f"Saving outputs to: {OUTPUT_DIR}")

## Name-cleaning rules

`MANUAL_ALIASES` contains only high-confidence variants found in the attached CSV. Add or remove entries here as you identify more historical name variants. The notebook deliberately does not merge names based only on approximate spelling.

In [ ]:
MANUAL_ALIASES = {
    # Full names versus initials
    "Brandt": "Brandt, Herbert W.",
    "Brandt, H. W.": "Brandt, Herbert W.",
    "Brandt, H.W.": "Brandt, Herbert W.",
    "Brand, H. W.": "Brandt, Herbert W.",
    "Brand, H.W.": "Brandt, Herbert W.",
    "Herbert W. Brandt": "Brandt, Herbert W.",
    "Congreve, W. M.": "Congreve, Major William Maitland",
    "ROSSIGNOL Jr., G. R.": "Rossignol Jr., Gilbert R.",
    "Scott, W. E.": "Scott, William Earle Dodge",
    "Wicks, M. L.": "Wicks Jr., M. L.",

    # Clear spelling or transcription variants
    "Jeffrys, C.": "Jefferys, C.",
    "Jeffries, C.": "Jefferys, C.",
    "Vaughn, R.": "Vaughan, R.",
    "Erstgate, T. F.": "Eastgate, T. F.",
    "Peyton, L. G.": "Payton, L. G.",
    "Eierce, A. K.": "Pierce, Alfred K.",
    "Moore, R. L.": "More, Robert L.",
}

EXCLUDE_UNKNOWN = False
TOP_N = 50
FULL_CHART_PANELS = 4

def ascii_words(value):
    value = unicodedata.normalize("NFKD", str(value))
    value = value.encode("ascii", "ignore").decode("ascii").casefold()
    value = value.replace("&", " and ")
    return re.findall(r"[a-z0-9]+", value)

def basic_key(value):
    return " ".join(ascii_words(value))

def person_token_key(value):
    # Exact token-set matching catches Given Surname versus Surname, Given.
    ignored = {"mr", "mrs", "ms", "dr", "reverend", "major"}
    tokens = [t for t in ascii_words(value) if t not in ignored]
    return tuple(sorted(tokens))

In [ ]:
df = pd.read_csv(CSV_PATH, low_memory=False)
required = {"Collector", "Individual Count"}
missing_columns = required.difference(df.columns)
if missing_columns:
    raise KeyError(f"Missing required columns: {sorted(missing_columns)}")

df["Egg Count"] = pd.to_numeric(df["Individual Count"], errors="coerce")
df["Collector Raw"] = df["Collector"].fillna("").astype(str).str.strip()

bad_counts = df["Egg Count"].isna()
blank_collectors = df["Collector Raw"].eq("")
print(f"Rows: {len(df):,}")
print(f"Physical eggs in rows with a usable count: {df['Egg Count'].sum():,.0f}")
print(f"Rows excluded for missing/unusable Individual Count: {bad_counts.sum():,}")
print(f"Rows excluded for blank Collector: {blank_collectors.sum():,}")

In [ ]:
# Standalone entries let us safely recognize reversed names inside compound fields.
raw_frequency = df.loc[~blank_collectors, "Collector Raw"].value_counts()
standalone = [
    name for name in raw_frequency.index
    if not re.search(r"\s+(?:and|&)\s+|;|\bet al\.?\b", name, flags=re.I)
]

token_candidates = defaultdict(list)
for name in standalone:
    token_candidates[person_token_key(name)].append(name)

# If several standalone labels have identical tokens, prefer the most frequent.
token_lookup = {
    key: max(names, key=lambda n: raw_frequency.get(n, 0))
    for key, names in token_candidates.items()
}

alias_lookup = {basic_key(source): target for source, target in MANUAL_ALIASES.items()}

def clean_fragment(fragment):
    fragment = fragment.strip(" ,.;")
    fragment = re.sub(r"^(?:for|by)\s+", "", fragment, flags=re.I).strip()
    if not fragment:
        return None

    # Apply explicit aliases before and after order normalization.
    if basic_key(fragment) in alias_lookup:
        return alias_lookup[basic_key(fragment)]

    reordered = token_lookup.get(person_token_key(fragment), fragment)
    return alias_lookup.get(basic_key(reordered), reordered)

def split_collectors(raw):
    raw = re.sub(r"\bet al\.?\b", "", raw, flags=re.I).strip(" ,.;")
    pieces = re.split(r"\s+(?:and|&)\s+|;", raw, flags=re.I)
    cleaned = [clean_fragment(piece) for piece in pieces]
    # Deduplicate within one record without changing order.
    return list(dict.fromkeys(name for name in cleaned if name))

In [ ]:
usable = df.loc[~bad_counts & ~blank_collectors, ["Collector Raw", "Egg Count"]].copy()
usable["Parsed Collector"] = usable["Collector Raw"].map(split_collectors)
credits = usable.explode("Parsed Collector", ignore_index=True)
credits = credits.dropna(subset=["Parsed Collector"])

if EXCLUDE_UNKNOWN:
    credits = credits[~credits["Parsed Collector"].str.contains("unknown", case=False, na=False)]

collector_totals = (
    credits.groupby("Parsed Collector", as_index=False)
    .agg(Eggs=("Egg Count", "sum"), Records=("Egg Count", "size"))
    .sort_values(["Eggs", "Parsed Collector"], ascending=[False, True])
    .reset_index(drop=True)
)
collector_totals["Eggs"] = collector_totals["Eggs"].astype(int)

print(f"Collectors after cleaning: {len(collector_totals):,}")
print(f"Egg credits after splitting co-collectors: {collector_totals['Eggs'].sum():,}")
display(collector_totals.head(25))

## Readable overview: top collectors

This linear-scale chart is the easiest overview. The second chart below includes every collector.

In [ ]:
top = collector_totals.head(TOP_N).sort_values("Eggs", ascending=True)
fig, ax = plt.subplots(figsize=(13, max(10, TOP_N * 0.31)))
bars = ax.barh(top["Parsed Collector"], top["Eggs"], color="#356A9A")
ax.bar_label(bars, labels=[f"{v:,}" for v in top["Eggs"]], padding=3, fontsize=8)
ax.set_title(f"Top {min(TOP_N, len(top))} collectors by number of eggs", loc="left", fontsize=16, weight="bold")
ax.set_xlabel("Eggs credited")
ax.set_ylabel("")
ax.grid(axis="x", color="#D9DEE3", linewidth=0.8)
ax.set_axisbelow(True)
ax.spines[["top", "right", "left"]].set_visible(False)
ax.margins(x=0.10)
fig.tight_layout()
top_path = OUTPUT_DIR / "Eggs_by_collector_top50.png"
fig.savefig(top_path, dpi=200, bbox_inches="tight", facecolor="white")
plt.show()
print(f"Saved: {top_path}")

## Every collector

Because the collection is extremely uneven, this complete chart uses a logarithmic X axis. Otherwise Brandt's bar would make hundreds of small collectors visually disappear. Collectors remain sorted from largest to smallest across panels.

In [ ]:
n = len(collector_totals)
n_panels = min(FULL_CHART_PANELS, max(1, n))
rows_per_panel = math.ceil(n / n_panels)
fig_height = max(12, rows_per_panel * 0.26)
fig, axes = plt.subplots(1, n_panels, figsize=(6.3 * n_panels, fig_height), sharex=True)
if n_panels == 1:
    axes = [axes]

xmax = collector_totals["Eggs"].max() * 1.65
for panel, ax in enumerate(axes):
    start = panel * rows_per_panel
    stop = min((panel + 1) * rows_per_panel, n)
    part = collector_totals.iloc[start:stop].copy()
    y = range(len(part))
    bars = ax.barh(y, part["Eggs"], color="#356A9A", height=0.76)
    ax.set_yticks(list(y), labels=part["Parsed Collector"], fontsize=6.2)
    ax.invert_yaxis()
    ax.set_xscale("log")
    ax.set_xlim(0.8, xmax)
    ax.xaxis.set_major_locator(LogLocator(base=10))
    ax.xaxis.set_major_formatter(FuncFormatter(lambda x, pos: f"{x:,.0f}" if x >= 1 else ""))
    ax.grid(axis="x", which="both", color="#D9DEE3", linewidth=0.65)
    ax.set_axisbelow(True)
    ax.spines[["top", "right", "left"]].set_visible(False)
    ax.tick_params(axis="y", length=0, pad=3)
    for bar, value in zip(bars, part["Eggs"]):
        ax.text(value * 1.06, bar.get_y() + bar.get_height()/2, f"{value:,}", va="center", fontsize=5.7)
    ax.set_title(f"Ranks {start + 1:,}–{stop:,}", loc="left", fontsize=11, weight="bold")
    ax.set_xlabel("Eggs credited (log scale)")

fig.suptitle("Eggs credited to every collector", x=0.01, y=1.002, ha="left", fontsize=18, weight="bold")
fig.tight_layout(rect=(0, 0, 1, 0.998), w_pad=2.2)
all_path = OUTPUT_DIR / "Eggs_by_collector_all.png"
fig.savefig(all_path, dpi=150, bbox_inches="tight", facecolor="white")
plt.show()
print(f"Saved: {all_path}")

## Audit files and possible aliases

The audit shows how every raw field was split and labeled. The possible-alias file lists similar names for human review; it does **not** merge them automatically.

In [ ]:
audit = (
    credits.groupby(["Collector Raw", "Parsed Collector"], as_index=False)
    .agg(Records=("Egg Count", "size"), Eggs_Credited=("Egg Count", "sum"))
    .sort_values(["Parsed Collector", "Collector Raw"])
)
audit["Eggs_Credited"] = audit["Eggs_Credited"].astype(int)
audit_path = OUTPUT_DIR / "Collector_name_audit.csv"
totals_path = OUTPUT_DIR / "Eggs_by_collector_totals.csv"
audit.to_csv(audit_path, index=False)
collector_totals.to_csv(totals_path, index=False)

names = collector_totals["Parsed Collector"].tolist()
egg_lookup = collector_totals.set_index("Parsed Collector")["Eggs"].to_dict()
candidate_rows = []
for left, right in combinations(names, 2):
    left_key, right_key = basic_key(left), basic_key(right)
    if abs(len(left_key) - len(right_key)) > 15:
        continue
    similarity = SequenceMatcher(None, left_key, right_key).ratio()
    if similarity >= 0.88:
        candidate_rows.append({
            "Name 1": left, "Eggs 1": egg_lookup[left],
            "Name 2": right, "Eggs 2": egg_lookup[right],
            "Similarity": round(similarity, 3),
        })
possible_aliases = pd.DataFrame(candidate_rows).sort_values("Similarity", ascending=False) if candidate_rows else pd.DataFrame()
alias_review_path = OUTPUT_DIR / "Collector_possible_aliases_review.csv"
possible_aliases.to_csv(alias_review_path, index=False)

print("Saved:")
for path in [totals_path, audit_path, alias_review_path]:
    print(f"  {path}")
display(possible_aliases.head(30))